In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

class RandomForestAeroFitter:
    def __init__(self, excel_file='re_dataset.xlsx'):
        """
        Initialize the Random Forest fitter with data from Excel file
        Expected columns: Re, Cl, alpha
        """
        self.data = pd.read_excel(excel_file)
        self.prepare_data()
        
    def prepare_data(self):
        """Prepare and validate the data"""
        print("Data shape:", self.data.shape)
        print("Columns:", self.data.columns.tolist())
        print("Data preview:")
        print(self.data.head())
        print("\nData ranges:")
        print(f"Reynolds number: {self.data['Re'].min():.0f} to {self.data['Re'].max():.0f}")
        print(f"Alpha (degrees): {self.data['alpha'].min():.1f} to {self.data['alpha'].max():.1f}")
        print(f"Cl: {self.data['Cl'].min():.3f} to {self.data['Cl'].max():.3f}")
        
        # Prepare features (X) and target (y)
        self.X = self.data[['Re', 'alpha']].values
        self.y = self.data['Cl'].values
        
        # Create train-test split
        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
            self.X, self.y, test_size=0.2, random_state=42
        )
        
    def fit_random_forest(self, n_estimators=100, max_depth=None, min_samples_split=2):
        """Train Random Forest model"""
        print("\n=== Training Random Forest ===")
        
        self.rf_model = RandomForestRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            random_state=42,
            n_jobs=-1
        )
        
        # Fit the model
        self.rf_model.fit(self.X_train, self.y_train)
        
        # Make predictions
        y_pred_train = self.rf_model.predict(self.X_train)
        y_pred_test = self.rf_model.predict(self.X_test)
        
        # Calculate metrics
        train_mse = mean_squared_error(self.y_train, y_pred_train)
        test_mse = mean_squared_error(self.y_test, y_pred_test)
        train_r2 = r2_score(self.y_train, y_pred_train)
        test_r2 = r2_score(self.y_test, y_pred_test)
        
        print(f"Training MSE: {train_mse:.6f}")
        print(f"Test MSE: {test_mse:.6f}")
        print(f"Training R²: {train_r2:.6f}")
        print(f"Test R²: {test_r2:.6f}")
        
        # Feature importance
        feature_importance = self.rf_model.feature_importances_
        print(f"\nFeature Importance:")
        print(f"Reynolds Number: {feature_importance[0]:.3f}")
        print(f"Angle of Attack: {feature_importance[1]:.3f}")
        
        self.performance = {
            'train_mse': train_mse,
            'test_mse': test_mse,
            'train_r2': train_r2,
            'test_r2': test_r2
        }
        
        return self.rf_model
    
    def create_prediction_function(self):
        """Create a callable function for Cl predictions"""
        if not hasattr(self, 'rf_model'):
            print("Please train the model first using fit_random_forest()!")
            return None
        
        def predict_cl(reynolds, alpha):
            """
            Predict Cl coefficient for given Reynolds number and angle of attack
            
            Parameters:
            reynolds: float or array-like, Reynolds number
            alpha: float or array-like, angle of attack in degrees
            
            Returns:
            Cl: float or array, coefficient of lift
            
            Example:
            cl = predict_cl(1000000, 5.0)  # Single prediction
            cl = predict_cl([1000000, 2000000], [5.0, 8.0])  # Multiple predictions
            """
            # Handle single values or arrays
            reynolds = np.atleast_1d(reynolds)
            alpha = np.atleast_1d(alpha)
            
            if len(reynolds) == 1 and len(alpha) > 1:
                reynolds = np.full_like(alpha, reynolds[0])
            elif len(alpha) == 1 and len(reynolds) > 1:
                alpha = np.full_like(reynolds, alpha[0])
            elif len(reynolds) != len(alpha):
                raise ValueError("Reynolds and alpha must have the same length or one must be scalar")
            
            # Check if values are within training range
            re_min, re_max = self.data['Re'].min(), self.data['Re'].max()
            alpha_min, alpha_max = self.data['alpha'].min(), self.data['alpha'].max()
            
            if np.any(reynolds < re_min) or np.any(reynolds > re_max):
                print(f"Warning: Reynolds number outside training range [{re_min:.0f}, {re_max:.0f}]")
            if np.any(alpha < alpha_min) or np.any(alpha > alpha_max):
                print(f"Warning: Alpha outside training range [{alpha_min:.1f}, {alpha_max:.1f}]")
            
            # Create input array and predict
            X_pred = np.column_stack([reynolds, alpha])
            Cl_pred = self.rf_model.predict(X_pred)
            
            # Return scalar if input was scalar
            if len(Cl_pred) == 1:
                return float(Cl_pred[0])
            return Cl_pred
        
        # Add metadata to the function
        predict_cl.model_performance = self.performance
        predict_cl.data_ranges = {
            'reynolds': (self.data['Re'].min(), self.data['Re'].max()),
            'alpha': (self.data['alpha'].min(), self.data['alpha'].max()),
            'cl': (self.data['Cl'].min(), self.data['Cl'].max())
        }
        
        print(f"\nPrediction function created!")
        print(f"Model performance - Test R²: {self.performance['test_r2']:.4f}")
        print(f"Usage: cl = predict_cl(reynolds_number, angle_of_attack)")
        
        return predict_cl
    
    def plot_surface(self, grid_size=50):
        """Plot 3D surface of the Random Forest model"""
        if not hasattr(self, 'rf_model'):
            print("Please train the model first using fit_random_forest()!")
            return None
            
        # Create grid for surface
        re_range = np.linspace(self.data['Re'].min(), self.data['Re'].max(), grid_size)
        alpha_range = np.linspace(self.data['alpha'].min(), self.data['alpha'].max(), grid_size)
        Re_grid, Alpha_grid = np.meshgrid(re_range, alpha_range)
        
        # Prepare grid points for prediction
        grid_points = np.column_stack([Re_grid.ravel(), Alpha_grid.ravel()])
        Cl_pred = self.rf_model.predict(grid_points)
        Cl_grid = Cl_pred.reshape(Re_grid.shape)
        
        # Create 3D plot
        fig = plt.figure(figsize=(12, 8))
        ax = fig.add_subplot(111, projection='3d')
        
        # Plot surface
        surf = ax.plot_surface(Re_grid, Alpha_grid, Cl_grid, cmap='viridis', 
                              alpha=0.8, edgecolor='none')
        
        # Plot original data points
        ax.scatter(self.data['Re'], self.data['alpha'], self.data['Cl'], 
                  c='red', s=20, alpha=0.8, label='Training data')
        
        ax.set_xlabel('Reynolds Number')
        ax.set_ylabel('Angle of Attack (degrees)')
        ax.set_zlabel('Coefficient of Lift (Cl)')
        ax.set_title(f'Cl Surface - Random Forest (R² = {self.performance["test_r2"]:.4f})')
        
        # Add colorbar
        fig.colorbar(surf, shrink=0.5, aspect=5)
        ax.legend()
        
        plt.tight_layout()
        plt.show()
        
        return fig
    
    def test_predictions(self, test_cases=None):
        """Test the prediction function with some example cases"""
        if not hasattr(self, 'rf_model'):
            print("Please train the model first!")
            return
        
        predict_cl = self.create_prediction_function()
        
        if test_cases is None:
            # Create some test cases within the data range
            re_mid = (self.data['Re'].min() + self.data['Re'].max()) / 2
            alpha_mid = (self.data['alpha'].min() + self.data['alpha'].max()) / 2
            
            test_cases = [
                (self.data['Re'].min(), self.data['alpha'].min()),
                (re_mid, alpha_mid),
                (self.data['Re'].max(), self.data['alpha'].max()),
            ]
        
        print(f"\n=== Testing Predictions ===")
        print(f"{'Reynolds':<12} {'Alpha':<8} {'Predicted Cl':<12}")
        print("-" * 35)
        
        for re, alpha in test_cases:
            cl = predict_cl(re, alpha)
            print(f"{re:<12.0f} {alpha:<8.1f} {cl:<12.4f}")

# Usage example
if __name__ == "__main__":
    # Create the fitter object
    fitter = RandomForestAeroFitter('re_dataset.xlsx')
    
    # Train the Random Forest model
    rf_model = fitter.fit_random_forest(n_estimators=200)
    
    # Create the prediction function
    predict_cl = fitter.create_prediction_function()
    
    # Test some predictions
    fitter.test_predictions()
    
    # Plot the 3D surface
    fitter.plot_surface()
    
    # Now you can use predict_cl anywhere in your code:
    print(f"\n=== Example Usage ===")
    print(f"Single prediction:")
    cl_single = predict_cl(1500000, 7.5)
    print(f"Cl at Re=1,500,000 and alpha=7.5°: {cl_single:.4f}")
    
    print(f"\nMultiple predictions:")
    reynolds_list = [1000000, 1500000, 2000000]
    alpha_list = [5.0, 7.5, 10.0]
    cl_multiple = predict_cl(reynolds_list, alpha_list)
    for re, alpha, cl in zip(reynolds_list, alpha_list, cl_multiple):
        print(f"Re={re:.0f}, alpha={alpha:.1f}° → Cl={cl:.4f}")

FileNotFoundError: [Errno 2] No such file or directory: 're_dataset.xlsx'